### **Data Reading**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('parquet').load("abfss://bronze@databrcks.dfs.core.windows.net/products")
display(df.limit(10))

product_id,product_name,category,brand,price,_rescued_data
P0001,Clearly Its,Beauty,Nike,1868.54,null
P0002,Production Clear,Beauty,Apple,587.13,null
P0003,Culture Coach,Home,Revlon,1599.24,null
P0004,Movement Part,Sports,LG,651.71,null
P0005,Fact Name,Clothing,Samsung,1861.78,null
P0006,Usually Stop,Toys,Adidas,936.36,null
P0007,Reveal Current,Sports,Adidas,1954.02,null
P0008,Force Language,Beauty,Puma,1251.26,null
P0009,Stage Leg,Clothing,Samsung,1247.15,null
P0010,Leader Then,Sports,Sony,975.53,null


In [0]:
df = df.drop('_rescued_data')

###**Reusable functions**

In [0]:
df.createOrReplaceTempView("products")


In [0]:
%sql
create or replace function portfolio_project.bronze.discounted_price(price double)
returns double
language sql
return 
    case 
        when price > 1500 then price * 0.9 
        when price between 500 and 1500 then price * 0.7
        else price * 0.5
    end

In [0]:
df = df.withColumn("discounted_price", expr('portfolio_project.bronze.discounted_price(price)'))
display(df)

product_id,product_name,category,brand,price,discounted_price
P0001,Clearly Its,Beauty,Nike,1868.54,1681.686
P0002,Production Clear,Beauty,Apple,587.13,410.991
P0003,Culture Coach,Home,Revlon,1599.24,1439.316
P0004,Movement Part,Sports,LG,651.71,456.197
P0005,Fact Name,Clothing,Samsung,1861.78,1675.602
P0006,Usually Stop,Toys,Adidas,936.36,655.452
P0007,Reveal Current,Sports,Adidas,1954.02,1758.618
P0008,Force Language,Beauty,Puma,1251.26,875.882
P0009,Stage Leg,Clothing,Samsung,1247.15,873.005
P0010,Leader Then,Sports,Sony,975.53,682.871


In [0]:
%sql
create or replace function portfolio_project.bronze.upper_func(brand STRING)
returns string
language python
as
$$
    return brand.upper()
$$

In [0]:
df = df.withColumn('brand_name', expr('portfolio_project.bronze.upper_func(brand)'))
df = df.drop('brand')
display(df)

product_id,product_name,category,price,discounted_price,brand_name
P0001,Clearly Its,Beauty,1868.54,1681.686,NIKE
P0002,Production Clear,Beauty,587.13,410.991,APPLE
P0003,Culture Coach,Home,1599.24,1439.316,REVLON
P0004,Movement Part,Sports,651.71,456.197,LG
P0005,Fact Name,Clothing,1861.78,1675.602,SAMSUNG
P0006,Usually Stop,Toys,936.36,655.452,ADIDAS
P0007,Reveal Current,Sports,1954.02,1758.618,ADIDAS
P0008,Force Language,Beauty,1251.26,875.882,PUMA
P0009,Stage Leg,Clothing,1247.15,873.005,SAMSUNG
P0010,Leader Then,Sports,975.53,682.871,SONY


###**Data Writing**


In [0]:
df.write.mode('append').format('delta').save('abfss://silver@databrcks.dfs.core.windows.net/products')